| 3 | DeepSeek agent (IPP loop + tools) | `tools.engine` / `tools.graph_tools` / `LLMs.deepseek` |

## 1. Setup & Imports

In [ ]:
from tools.encoder import EncoderLayer
from tools.ipp import IPP, ToolRegistry
from tools.build import build_graph, export_backward_compatible
from tools.graph_tools import ensure_tools
from tools.agents import NodeAgent, GrowthAgent
from LLMs.deepseek import DeepSeekProvider, MockProvider

print('workspace :', Config.WORKSPACE_ROOT)
print('assets    :', Config.ASSETS_DIR)
print('env       : agentic_ai (conda)')
ensure_tools()
print('tools     :', len(ToolRegistry.names()), 'registered')
print('llm       :', Config.get_model(), '->', Config.DEEPSEEK_BASE_URL)

workspace : C:\Deepin\Programming\20260720 GraphRAG
assets    : C:\Deepin\Programming\20260720 GraphRAG\assets
env       : agentic_ai (conda)
tools     : 18 registered
llm       : deepseek-v4-flash -> https://api.deepseek.com


## 2. Build the Global Graph from `assets/`

Seed nodes: the **three survey papers** (Peng / Han / Yang) + **20 concept nodes** from their content, with typed edges and §4.3a bidirectional consistency. Paper chunks are extracted from `assets/extracted/` and embedded by the encoder layer.

In [2]:
t0 = time.time()
graph, encoder = build_graph()   # idempotent: safe to re-run
print(graph.summary())
print(f'chunks indexed : {encoder.index.size()}')
print(f'consistency    : {len(graph.validate_consistency())} violations (§4.3a)')
print(f'components     : {len(graph.connected_components())}')
print(f'build time     : {time.time()-t0:.1f}s')

export_backward_compatible(graph)
print('\nbackward-compatible export ->', Config.GRAPH_DIR / 'export' / 'structurelist.json')

KnowledgeGraph: 23 nodes, 31 edges
density=0.1225  components=1
  pagerank[Graph-based Agent Memory (Yang et al. 2026)]=0.0904
  pagerank[GraphRAG with Graphs (Han et al. 2025)]=0.0861
  pagerank[GraphRAG Framework]=0.0833
  pagerank[G-Retrieval]=0.0612
  pagerank[Graph RAG Survey (Peng et al. 2024)]=0.0597
  pagerank[Memory Evolution]=0.0496
chunks indexed : 802
consistency    : 0 violations (§4.3a)
components     : 1
build time     : 0.5s

backward-compatible export -> C:\Deepin\Programming\20260720 GraphRAG\graph_data\export\structurelist.json


## 3. Local Graphs — Depth-3 Working Memory

**The core requirement**: every node materializes its depth-3 connected neighborhood. Agents never see the global graph; they operate on $\mathcal{L}_3(u)$.

In [3]:
for anchor in ['agent_memory', 'g_retrieval', 'peng_survey']:
    local = graph.materialize_local(anchor, depth=3)
    print(f'\n=== L3({graph.get_node(anchor).entryname}) ===')
    print('  stats:', local.stats)
    print('  nodes:', [n.entryname for n in list(local.nodes.values())[:14]])
    print('  sample path:', [graph.get_node(n).entryname for n in (local.paths[0] if local.paths else [])])


=== L3(Agent Memory) ===
  stats: {'n': 13, 'm': 14, 'density': 0.1794871794871795, 'diameter': 3}
  nodes: ['Hybrid RAG', 'Graph RAG Survey (Peng et al. 2024)', 'Memory Storage', 'DeepSeek Agent', 'Community Summaries', 'Memory Extraction', 'Agent Memory', 'G-Retrieval', 'Local Graphs (depth-3)', 'Self-Improvement Loop', 'Graph-based Agent Memory (Yang et al. 2026)', 'Memory Evolution', 'Memory Retrieval']
  sample path: ['Agent Memory', 'Local Graphs (depth-3)', 'G-Retrieval', 'Hybrid RAG']

=== L3(G-Retrieval) ===
  stats: {'n': 19, 'm': 27, 'density': 0.15789473684210525, 'diameter': 3}
  nodes: ['Retriever', 'G-Indexing', 'Memory Evolution', 'GraphRAG Framework', 'Graph RAG Survey (Peng et al. 2024)', 'Community Summaries', 'Agent Memory', 'Graph-based Agent Memory (Yang et al. 2026)', 'G-Generation', 'GraphRAG with Graphs (Han et al. 2025)', 'DeepSeek Agent', 'Query Processor', 'Local Graphs (depth-3)', 'Self-Improvement Loop']
  sample path: ['G-Retrieval', 'Graph RAG Survey (P

## 4. Encoder Layer — Vector RAG Extraction

The encoder gives every node a **vector address**: chunks of paper content are embedded and searchable. This is the *encoder-like capability for vector-based RAG extraction into the nodes*.

In [4]:
queries = [
    'community detection and hierarchical summaries for global questions',
    'graph neural network message passing retrieval',
    'temporal knowledge graph conflict resolution',
]
for q in queries:
    print(f'\nQ: {q}')
    hits = encoder.search(q, k=3)
    for chunk, sim in hits:
        print(f'   {chunk.chunk_id} [{chunk.section}] sim={sim:.3f}: {chunk.text[:90]}...')
print('\nhybrid node ranking for "self-improving memory":')
for nid, score in encoder.hybrid_search('self-improving memory', graph, 'agent_memory', k=5):
    print(f'   {graph.get_node(nid).entryname}  score={score:.3f}')


Q: community detection and hierarchical summaries for global questions
   vec://community_summary/meta [__meta__] sim=0.627: Community Summaries Leiden communities + LLM summaries for global questions (Microsoft Gra...
   vec://yang_survey/c0628 [page-9] sim=0.395: tree is a directed acyclic graph (DAG) that explicitly
models parent-child and containment...
   vec://han_survey/c0222 [page-9] sim=0.380: match the recognized types for next-round exploration. For example, given the question "Wh...

Q: graph neural network message passing retrieval
   vec://peng_survey/c0041 [page-12] sim=0.408: Graph Retrieval-Augmented Generation: A Survey
111:11
Input Query
§ 6.4.1 Query Enhancemen...
   vec://yang_survey/c0658 [page-13] sim=0.401: retrieval, which increases search depth and coverage
via repeated retrieval; ii) post-retr...
   vec://peng_survey/c0046 [page-13] sim=0.368: ficant computational overhead. Considering this complementarity, many
methods propose hybr...

Q: temporal knowledge

## 5. The DeepSeek Agent Operating on a Node (IPP loop)

The **NodeAgent** materializes $\mathcal{L}_3(u)$, pulls encoded evidence, grounds the task in them, and runs the DeepSeek Chat Completions agent loop (function calling → four-phase tool pipeline → answer). If the API is unreachable, a deterministic `MockProvider` keeps the pipeline demonstrable.

In [5]:
# Pick a provider: real DeepSeek, falling back to the offline MockProvider
try:
    llm = DeepSeekProvider(model=Config.get_model())
    llm.chat([{'role':'user','content':'Reply: OK'}], max_tokens=8)
    provider_name = f'deepseek:{llm.model}'
except Exception as exc:
    llm = MockProvider()
    provider_name = 'mock (offline fallback)'
print('provider:', provider_name)

node_agent = NodeAgent(graph, encoder, llm=llm)
t0 = time.time()
result = node_agent.operate(
    'g_retrieval',
    'List the concrete retrieval techniques mentioned in the local graph and how they relate.',
)
print('\n=== ANSWER ===')
print(result['answer'][:1600])
print('\n=== WORKING MEMORY ===', result['local_graph'])
print('=== TOOLS CALLED ===', [e['tool'] for e in result['trace'] if e['tool']])
print(f'=== run time: {time.time()-t0:.1f}s, tokens: {result["tokens"]}')

provider: deepseek:deepseek-v4-flash



=== ANSWER ===
Based on the local graph of G-Retrieval, here are the concrete retrieval techniques and their relationships:

## Concrete Retrieval Techniques

1. **Local Graphs (depth-3)** — Depth-k ego networks used as bounded working memory for agents; enables multi-hop reasoning. *G-Retrieval uses this.*

2. **Hybrid RAG** — Combines vector similarity retrieval with graph traversal (HybridRAG pattern). *G-Retrieval uses this.*

3. **Community Summaries** — Leiden communities + LLM summaries for global questions (Microsoft GraphRAG). *G-Retrieval uses this.*

4. **Encoder Layer (vector RAG)** — Chunk → embed → vector index → hybrid (similarity ⊕ structure) retrieval. *This enables Hybrid RAG.*

5. **Retriever** (concept) — Heuristic/LM/GNN/agent-based retrieval; k-hop, shortest-path, PCST, communities. This is the broader retrieval component within the GraphRAG Framework.

## How They Relate

- **G-Retrieval** is the graph-guided retrieval node (nodes/triples/paths/subgraphs at once

## 6. Growth Agent — Recursive Self-Improvement (Layer 4)

The **GrowthAgent** implements the recursive growth loop:

1. **Probe** — detect gaps in $\mathcal{L}_3(u)$ (isolation, components, missing descriptions)
2. **Propose** — DeepSeek returns a structured JSON growth proposal grounded in the local graph + encoder evidence
3. **Apply** — programmatic application through graph tools with **dedup**, **per-run limits** (≤3 nodes here), and **§4.3a consistency**
4. **Validate + persist** — run log (VCL) + graph save

In [6]:
growth = GrowthAgent(graph, encoder, llm=llm)
before = graph._nodes.copy()
t0 = time.time()
growth_result = growth.expand(
    'agent_memory',
    'Graph Memory Benchmarks',
    'add knowledge about benchmarks for evaluating graph-based agent memory',
)
print('=== PROPOSAL ===')
print(json.dumps(growth_result['proposal'], ensure_ascii=False, indent=1)[:900])
print('\n=== APPLIED NODES ===')
for n in growth_result['applied_nodes']:
    print('  +', n['node_id'], '-', n['entryname'])
print('=== APPLIED EDGES ===')
for e in growth_result['applied_edges'][:10]:
    print('  ', e['source'], '--[' + e['relation'] + ']-->', e['target'])
print('\nskipped  :', growth_result['skipped'])
print('errors   :', growth_result['errors'])
print('violations:', growth_result['consistency_violations'])
print(f'run time : {time.time()-t0:.1f}s, nodes {len(before)} -> {len(graph._nodes)}')
print('\nlast run log entry (VCL):')
print(json.dumps(graph.runs()[-1], ensure_ascii=False, indent=1)[:500])

=== PROPOSAL ===
{
 "new_nodes": [
  {
   "node_id": "Graph Memory Benchmarks",
   "entryname": "Graph Memory Benchmarks",
   "category": "concept",
   "description": "Benchmarks for evaluating graph-based agent memory systems, focusing on retrieval accuracy, reasoning capability, and memory evolution over time.",
   "links": [
    {
     "source": "Graph Memory Benchmarks",
     "target": "Agent Memory",
     "relation": "describes"
    }
   ]
  },
  {
   "node_id": "Benchmark Datasets",
   "entryname": "Benchmark Datasets",
   "category": "concept",
   "description": "Standardized datasets used to test graph memory performance, including synthetic and real-world graphs with temporal and relational complexity.",
   "links": [
    {
     "source": "Benchmark Datasets",
     "target": "Graph Memory Benchmarks",
     "relation": "part_of"
    }
   ]
  },
  {
   "node_id": "Evaluation Metrics",
   "entrynam

=== APPLIED NODES ===
  + Graph Memory Benchmarks - Graph Memory Benchmarks
  + B

## 7. The Grown Graph & What Happened

After one growth iteration the graph is denser and the anchor's local graph covers the new benchmark knowledge. Verify the updated topology and the backward-compatible export.

In [7]:
print(graph.summary())
local = graph.materialize_local('agent_memory', depth=3)
print(f'\nL3(agent_memory) now: {local.stats}')
print('new nodes in local graph:', [n.entryname for n in local.nodes.values() if 'benchmark' in n.node_id])
print('\nbackward-compatible structurelist.json (first 6):')
reg = json.loads((Config.GRAPH_DIR / 'export' / 'structurelist.json').read_text(encoding='utf-8'))
print(json.dumps(reg[:6], ensure_ascii=False, indent=1))

KnowledgeGraph: 26 nodes, 38 edges
density=0.1169  components=1
  pagerank[GraphRAG with Graphs (Han et al. 2025)]=0.0760
  pagerank[GraphRAG Framework]=0.0734
  pagerank[Graph-based Agent Memory (Yang et al. 2026)]=0.0629
  pagerank[Graph Memory Benchmarks]=0.0602
  pagerank[G-Retrieval]=0.0524
  pagerank[Graph RAG Survey (Peng et al. 2024)]=0.0521

L3(agent_memory) now: {'n': 16, 'm': 21, 'density': 0.175, 'diameter': 3}
new nodes in local graph: []

backward-compatible structurelist.json (first 6):
[
 {
  "folderid": "agent_memory",
  "entryname": "Agent Memory"
 },
 {
  "folderid": "community_summary",
  "entryname": "Community Summaries"
 },
 {
  "folderid": "deepseek_agent",
  "entryname": "DeepSeek Agent"
 },
 {
  "folderid": "encoder_layer",
  "entryname": "Encoder Layer (vector RAG)"
 },
 {
  "folderid": "g_generation",
  "entryname": "G-Generation"
 },
 {
  "folderid": "g_indexing",
  "entryname": "G-Indexing"
 }
]


---

## 8. Web Control Center (`ui/`)

The project ships a self-contained web interface (Flask REST API + vanilla-JS
SPA, no build step) that serves at **http://127.0.0.3:8000**:

```bash
python ui/server.py                # → http://127.0.0.3:8000
python ui/server.py --port 5000    # → http://127.0.0.3:5000
```

The server auto-loads the persisted graph (`graph_data/`) or rebuilds it from
`assets/`, and exposes the full API:

| Method | Endpoint | Purpose |
|---|---|---|
| GET | `/api/graph/summary` | stats, pagerank top, backward-compatible registry |
| GET | `/api/graph/local/<id>?depth=3` | depth-k ego-network payload for viz |
| POST | `/api/search` | vector RAG over the encoder layer |
| POST | `/api/agent/node` | run the Node agent on a node's local graph |
| POST | `/api/agent/grow` | run the Growth agent (recursive self-improvement) |
| GET | `/api/runs` | VCL run log |

The frontend (`ui/static/`) provides a force-directed graph canvas (global view
or any node's local graph, depth 1–4), a node detail drawer, a semantic search
panel, agent controls, and a run-log viewer — all local-first with no external
CDN dependencies.

---

## Version Control Log

Per ScientificInfrastructure §4.4a.

### v1.10 — 2026-08-02 (true streaming agent output, Copilot agent)

- All agents now **stream their output progressively** instead of "stuck then dump".
- `DeepSeekProvider.chat_stream()` — a true streaming generator over the OpenAI/DeepSeek SDK (`stream=True`), yielding `("thinking", delta)`, `("text", delta)`, `("tool_delta", …)` as tokens arrive; returns the accumulated `LLMResult` via PEP 380.
- `AgentEngine.chat_stream()` consumes the streaming LLM call: yields `message_delta` events as assistant text streams word-by-word, plus full `thinking`/`message`/`tool_call`/`tool_result`/`text` events.
- New Flask endpoint **`POST /api/agent/chat/stream`** — Server-Sent Events; emits each `ToolCallEvent` as it happens.
- Frontend Agent tab consumes the SSE stream and renders events live: a streaming 💬 message step grows with each delta, 🛠 tool calls appear the moment they're invoked, and the final answer box grows progressively.
- Gradio chat updated to stream the process + answer progressively too.
- Verified: engine streams (message_delta ST→REAM→-→OK), SSE first event in 1.03s, browser showed 16 process steps mid-stream at 4s (still running) then 27 steps / 4869-char answer at completion; Gradio emitted 84 progressive chunks.

### v1.9 — 2026-08-02 (foldable agentic process + read-only chat + audit tools, Copilot agent)